# Process cleaning data for Predictive Dynamic Staffing for Food Delivery Operations

## Import Library

In [ ]:
import pandas as pd


## Load dataset

In [12]:
df = pd.read_csv('food_delivery_orders_dataset.csv')

In [13]:
df.head()

,order_id,customer_id,restaurant_id,driver_id,order_timestamp,order_date,order_hour,day_of_week,is_weekend,city,...,traffic_level,delivery_partner_experience_months,delivery_partner_rating,restaurant_preparation_time_minutes,estimated_delivery_time_minutes,actual_delivery_time_minutes,late_delivery,order_status,cancellation_reason,customer_rating
0,ORD_000001,CUS_00001,RES_0008,DRV_0075,2025-10-05 15:54:00,2025-10-05,15,Sunday,1,City_C,...,Medium,14,4.4,25,25,29.0,0.0,Completed,NaN,4.6
1,ORD_000002,CUS_00013,RES_0024,DRV_0131,2025-10-01 20:00:00,2025-10-01,20,Wednesday,0,City_A,...,High,44,4.7,12,20,24.0,0.0,Completed,NaN,5.0
2,ORD_000003,CUS_00049,RES_0013,DRV_0087,2025-10-07 18:24:00,2025-10-07,18,Tuesday,0,City_C,...,High,1,5.0,11,63,72.0,1.0,Completed,NaN,3.6
3,ORD_000004,CUS_00007,RES_0001,DRV_0177,2025-10-30 16:54:00,2025-10-30,16,Thursday,0,City_A,...,Low,50,4.3,12,23,31.0,1.0,Completed,NaN,4.1
4,ORD_000005,CUS_00325,RES_0029,DRV_0114,2025-10-15 18:34:00,2025-10-15,18,Wednesday,0,City_B,...,Medium,21,4.0,9,32,42.0,1.0,Completed,NaN,3.9


## Data Filtering and Time-Window Aggregation


In [14]:
order_counts = df['restaurant_id'].value_counts()
busy_restaurants = order_counts[order_counts > 4000].index
df_filtered = df[df['restaurant_id'].isin(busy_restaurants)].copy()

df_filtered['order_timestamp'] = pd.to_datetime(df_filtered['order_timestamp'], errors='coerce')

df_filtered = df_filtered.dropna(subset=['order_timestamp']).copy()

## Information and describe dataset

In [17]:
df_filtered.describe()

,order_timestamp,order_hour,is_weekend,customer_age,restaurant_rating,items_count,subtotal,discount_percent,tax_amount,service_fee,...,order_total,tip_amount,distance_km,delivery_partner_experience_months,delivery_partner_rating,restaurant_preparation_time_minutes,estimated_delivery_time_minutes,actual_delivery_time_minutes,late_delivery,customer_rating
count,13098,13098.000000,13098.000000,13098.000000,13098.000000,13098.000000,13098.000000,13098.000000,13098.00000,13098.000000,...,13098.000000,12874.000000,13098.000000,13098.000000,13098.000000,13098.000000,13098.000000,12874.000000,12874.000000,12874.000000
mean,2025-10-16 03:55:28.021071872,14.471828,0.272332,26.315315,3.997595,2.914949,26.225967,4.510231,3.50643,1.247486,...,33.258039,1.422175,4.704505,30.140938,4.490663,14.464346,29.765231,34.165527,0.376262,4.282787
min,2025-10-01 00:15:00,0.000000,0.000000,18.000000,3.900000,1.000000,7.650000,0.000000,0.86000,0.190000,...,7.260000,0.000000,0.800000,1.000000,3.200000,5.000000,11.000000,10.000000,0.000000,1.900000
25%,2025-10-08 13:20:15,11.000000,0.000000,20.000000,3.900000,1.000000,9.920000,0.000000,1.35250,0.540000,...,15.942500,0.000000,2.800000,15.000000,4.300000,10.000000,21.000000,25.000000,0.000000,3.900000
50%,2025-10-16 10:32:00,14.000000,0.000000,26.000000,3.900000,2.000000,19.685000,0.000000,2.68000,0.980000,...,26.570000,0.000000,4.100000,31.000000,4.500000,13.000000,27.000000,32.000000,0.000000,4.300000
75%,2025-10-23 14:56:45,19.000000,1.000000,31.000000,4.200000,4.000000,36.170000,10.000000,4.80000,1.660000,...,44.307500,2.150000,5.900000,44.000000,4.800000,18.000000,36.000000,41.000000,1.000000,4.700000
max,2025-10-30 23:54:00,23.000000,1.000000,51.000000,4.200000,10.000000,102.660000,20.000000,14.26000,6.530000,...,124.800000,16.670000,20.000000,60.000000,5.000000,43.000000,107.000000,156.000000,1.000000,5.000000
std,NaN,5.187911,0.445177,7.291170,0.140553,2.034393,18.486397,6.414776,2.48731,0.950007,...,21.243851,2.282274,2.727546,16.830728,0.341901,5.993671,11.917609,14.029328,0.484466,0.539594


In [18]:
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13098 entries, 3 to 49997
Data columns (total 37 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   order_id                             13098 non-null  object        
 1   customer_id                          13098 non-null  object        
 2   restaurant_id                        13098 non-null  object        
 3   driver_id                            13098 non-null  object        
 4   order_timestamp                      13098 non-null  datetime64[ns]
 5   order_date                           13098 non-null  object        
 6   order_hour                           13098 non-null  int64         
 7   day_of_week                          13098 non-null  object        
 8   is_weekend                           13098 non-null  int64         
 9   city                                 13098 non-null  object        
 10  delivery_area  

In [19]:
prep = df_filtered['restaurant_preparation_time_minutes'].dropna()
print("Mean:", prep.mean())
print("Std Dev:", prep.std())
print("Median:", prep.median())
print("Skewness:", prep.skew())

Mean: 14.464345701633837
Std Dev: 5.993671100523908
Median: 13.0
Skewness: 0.8436649077598931


## Hourly Time-Window Aggregation and Data Imputation

In [20]:
windowed_df = (
    df_filtered.groupby(
        ['restaurant_id', pd.Grouper(key='order_timestamp', freq='1H')]
    )
    .agg(
        # TARGET & ANTRIAN
        demand=('order_id', 'count'),
        service_process_mean=('restaurant_preparation_time_minutes', 'mean'),

        # FITUR ML (X) - Temporal
        order_hour=('order_hour', 'first'),
        day_of_week=('day_of_week', 'first'),
        is_weekend=('is_weekend', 'first'),

        # FITUR ML (X) - Kondisi Eksternal
        weather=('weather', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),
        traffic_level=('traffic_level', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown'),

        # FITUR ML (X) - Karakteristik Pesanan
        total_items=('items_count', 'sum'),
        avg_subtotal=('subtotal', 'mean')
    )
    .reset_index()
)

# 3. Pembersihan nilai kosong akibat agregasi
windowed_df['service_process_mean'] = windowed_df.groupby('restaurant_id')['service_process_mean'].transform(lambda x: x.fillna(x.mean()))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_21140\1419244277.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  ['restaurant_id', pd.Grouper(key='order_timestamp', freq='1H')]


In [21]:
windowed_df.to_csv('cleaned_hourly_demand.csv', index=False)
windowed_df.head(10)

,restaurant_id,order_timestamp,demand,service_process_mean,order_hour,day_of_week,is_weekend,weather,traffic_level,total_items,avg_subtotal
0,RES_0001,2025-10-01 00:00:00,6,15.500000,0,Wednesday,0,Cloudy,Low,18,27.715000
1,RES_0001,2025-10-01 01:00:00,3,11.666667,1,Wednesday,0,Cloudy,Low,9,29.683333
2,RES_0001,2025-10-01 02:00:00,1,19.000000,2,Wednesday,0,Cloudy,Low,6,52.030000
3,RES_0001,2025-10-01 03:00:00,1,5.000000,3,Wednesday,0,Cloudy,Low,1,9.710000
4,RES_0001,2025-10-01 04:00:00,2,12.000000,4,Wednesday,0,Cloudy,Low,4,16.650000
5,RES_0001,2025-10-01 05:00:00,2,10.500000,5,Wednesday,0,Cloudy,Low,4,16.595000
6,RES_0001,2025-10-01 06:00:00,4,16.750000,6,Wednesday,0,Cloudy,High,17,39.310000
7,RES_0001,2025-10-01 07:00:00,5,8.800000,7,Wednesday,0,Cloudy,Low,5,9.366000
8,RES_0001,2025-10-01 08:00:00,10,13.400000,8,Wednesday,0,Cloudy,Medium,24,22.367000
9,RES_0001,2025-10-01 09:00:00,11,14.000000,9,Wednesday,0,Cloudy,Low,28,23.366364
